In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
import requests

In [ ]:
# -----------------------------
# Configuration
# -----------------------------
QUERY = "artificial intelligence automation emerging technology"
NUM_PAPERS = 2
FIELDS = "title,abstract,year,citationCount,influentialCitationCount,venue"
OUTPUT_FILE = "sources.txt"

# # -----------------------------
# # Semantic Scholar API call
# # -----------------------------
url = "https://api.semanticscholar.org/graph/v1/paper/search"
params = {
    "query": QUERY,
    "limit": NUM_PAPERS,
    "fields": FIELDS
}

response = requests.get(url, params=params)
#data = response.json()


class WeakTrainer:

    
    def __init__(self, model_name='all-MiniLM-L6-v2', use_forest=True):
        """
        Multi-output regression for field analysis
        
        Parameters:
        - model_name: sentence transformer model
        - use_forest: If True, uses Random Forest; if False, uses Ridge
        """
        self.embedding_model = SentenceTransformer(model_name)
        
        # Random Forest handles non-linear relationships better
        if use_forest:
            base_model = RandomForestRegressor(
                n_estimators=100,
                max_depth=10,
                random_state=42,
                n_jobs=-1
            )
        else:
            base_model = Ridge(alpha=1.0)
        
        self.regression_model = MultiOutputRegressor(base_model)
        self.is_fitted = False
        
        # Define dimensions
        self.dimensions = [
            'growth_potential',
            'recession_resistance', 
            'automation_resistance',
            'skill_accessibility',
            'cross_industry_collaboration',
            'saturation'
        ]

        self.growth_vector = np.sum(self.embedding_model.encode([
            "Rapidly ascending industry lifecycle",
            "High compound annual growth rate (CAGR)",
            "Successive waves of capital investment",
            "Proliferation of new business opportunities"
        ], convert_to_numpy=True, normalize_embeddings=True) - self.embedding_model.encode([
            "Terminal decline in consumer demand",
            "Market saturation and stagnating revenue",
            "Systemic contraction of the workforce",
            "Phasing out of legacy technologies"
        ], convert_to_numpy=True, normalize_embeddings=True), axis=0)

        self.automation_resistance_vector = np.sum(self.embedding_model.encode([
            "Navigating high-stakes ethical ambiguity",
            "Highly unstructured and unpredictable environments",
            "Complex interpersonal negotiation and conflict resolution",
            "Creative synthesis of disparate, non-linear ideas"
            "Needs humans"
        ], convert_to_numpy=True) - self.embedding_model.encode([
            "Rule-based decision making with clear logic",
            "Standardized procedures with predictable outcomes",
            "Quantitative analysis of structured datasets",
            "Mechanical execution of pre-defined instructions"
            "Does not require humans"
        ], convert_to_numpy=True), axis=0)

        self.recession_resistance_vector = np.mean(self.embedding_model.encode([
            "Non-discretionary spending required for survival",
            "Counter-cyclical demand that remains stable during downturns",
            "Fundamental human necessity regardless of economic climate",
            "Primary, abundant resource with no viable substitutes"
        ], convert_to_numpy=True, normalize_embeddings=True) - self.embedding_model.encode([
            "Luxury goods and non-essential premium services",
            "Postponable expenses during financial uncertainty",
            "Speculative assets with high volatility",
            "Resource sourced from very few places"
        ], convert_to_numpy=True, normalize_embeddings=True), axis=0)

        self.skill_accessibility_vector = np.mean(self.embedding_model.encode([
            "Rapid skill acquisition through self-directed study",
            "Democratized information available to the general public",
            "Low technical barrier to entry for beginners",
            "Short learning curve with immediate proficiency"
        ], convert_to_numpy=True, normalize_embeddings=True) - self.embedding_model.encode([
            "Protected institutional knowledge and trade secrets",
            "Stringent professional certification and licensing requirements",
            "High cognitive load involving advanced theoretical mastery",
            "Exclusive apprenticeship under master practitioners"
        ], convert_to_numpy=True, normalize_embeddings=True), axis=0)

        self.cross_industry_vector = np.mean(self.embedding_model.encode([
            "Transferable skills across diverse domains",
            "Interdisciplinary collaboration and cross-pollination",
            "Universal utility in both private and public sectors",
            "Foundational principle underlying multiple fields"
        ], convert_to_numpy=True, normalize_embeddings=True) - self.embedding_model.encode([
            "Isolated domain with no external dependencies",
            "Highly specific utility restricted to one industry",
            "Insular field of study with limited reach",
            "Siloed workflow with no cross-departmental impact"
        ], convert_to_numpy=True, normalize_embeddings=True), axis=0)
        
        self.growth_vector = self.growth_vector / np.linalg.norm(self.growth_vector)
        self.automation_resistance_vector = self.automation_resistance_vector / np.linalg.norm(self.automation_resistance_vector)
        self.recession_resistance_vector = self.recession_resistance_vector / np.linalg.norm(self.recession_resistance_vector)
        self.skill_accessibility_vector = self.skill_accessibility_vector / np.linalg.norm(self.skill_accessibility_vector)
        self.cross_industry_vector = self.cross_industry_vector / np.linalg.norm(self.cross_industry_vector)
    
    def cosine_similarity(self, text, anchor):
        """normalized vectors"""
        return np.dot(text, anchor) 
  

    def vectorize(self, statement):
        """
        Compute a simple score:
        - More recent papers → higher score
        - More citations → higher score
        """
        text = self.embedding_model.encode(statement, convert_to_numpy=True, normalize_embeddings=True)

        growth_factor = 10 * (self.cosine_similarity(text, 
        self.growth_vector)
    )
        automation_resistance = 10 * (self.cosine_similarity(text, 
        self.automation_resistance_vector)
    )
        recession_resistance = 10 * self.cosine_similarity(text,
        self.recession_resistance_vector
    )
        skill_accessibility = 10 * self.cosine_similarity(text,
        self.skill_accessibility_vector
    )
        cross_industry_collaboration = 10 * self.cosine_similarity(text,
        self.cross_industry_vector
    )
        return np.array([growth_factor, automation_resistance, recession_resistance, skill_accessibility, cross_industry_collaboration])

    def fileIFY(self, data):
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            for paper in data.get("data", []):
                score = self.vectorize(paper)
                f.write(f"Title: {paper.get('title')}\n")
                f.write(f"Score: {score}\n")
                abstract = paper.get("abstract", "No abstract available")
                f.write(f"Abstract: {abstract}\n")
                f.write("-" * 10 + "\n")



In [ ]:
rows = []

data = read("sources.txt")

print(data)

for paper in data.get("data", []):
    text = f"{paper.get('title','')} {paper.get('abstract','')}"
    
    row = {
        "text": text,
        "growth_potential": growth_score,
        "recession_resistance": recession_score,
        "automation_resistance": automation_score,
        "skill_accessibility": accessibility_score,
        "cross_industry_collaboration": collaboration_score
    }
    
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("auto_labeled_fields.csv", index=False)

<_io.TextIOWrapper name='sources.txt' mode='r' encoding='cp1252'>


AttributeError: '_io.TextIOWrapper' object has no attribute 'get'

In [4]:
embedding_model = WeakTrainer("all-MiniLM-L6-v2")

sentence = "AI requires expertise from hundreds of fields"
print(embedding_model.vectorize("This field is expected to grow in the coming future."))
print(embedding_model.vectorize("This field does not require human input"))
print(embedding_model.vectorize("This field is highly dependent on rare materials"))
print(embedding_model.vectorize("This field can be learned quickly online."))
print(embedding_model.vectorize("This field draws from the expertise of hundreds of fields."))



[ 1.696642   -0.4713351  -0.23263119  0.6209561  -0.9056866 ]
[-0.8512137  -0.66318226  1.2220014   1.2779863  -0.20865798]
[ 1.8080105   0.5395994   0.14776161 -0.38464373 -0.27441475]
[ 0.2585316 -0.6265162 -1.4481353  2.839861   0.6214818]
[ 0.47858888  0.4631634  -1.5185231   1.092448    0.5809562 ]
